# Sprint 1 — BluaDiagnostics

## IA Conversacional para Check-up Digital

PoC acadêmica desenvolvida para o Challenge BluaDiagnostics utilizando:

- Prompt Engineering;
- memória conversacional;
- function calling simulado;
- integração com Ollama Cloud.

O notebook demonstra:

- system prompt clínico;
- guardrails;
- tools simuladas;
- fluxo de suporte ao beneficiário;
- contexto multi-turno.


## 1. Instalação das dependências

In [ ]:
!pip install -q ollama ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 13.3 MB/s eta 0:00:00


## 2. Configuração do Ollama Cloud

No menu lateral esquerdo do Colab:

🔑 Secrets

Crie:

```text
OLLAMA_API_KEY
```

Cole sua chave do Ollama Cloud.


In [ ]:
from google.colab import userdata
from ollama import Client

OLLAMA_API_KEY = userdata.get("OLLAMA_API_KEY")

if not OLLAMA_API_KEY:
    raise ValueError("Configure o Secret OLLAMA_API_KEY no Colab.")

client = Client(
    host="https://ollama.com",
    headers={
        "Authorization": "Bearer " + OLLAMA_API_KEY
    }
)

MODEL_NAME = "gpt-oss:120b"

print("✅ Ollama Cloud configurado com sucesso")


✅ Ollama Cloud configurado com sucesso


## 3. Teste de conexão

In [ ]:
teste = client.chat(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Responda apenas OK"
        }
    ],
    stream=False
)

print("Resposta do modelo:")
print(teste["message"]["content"])


Resposta do modelo:
OK


## 4. System Prompt Clínico

O agente atua como assistente de apoio ao beneficiário da Care Plus.


In [ ]:
system_prompt = """
Você é o BluaDiagnostics, um assistente conversacional da Care Plus.

OBJETIVO:
Auxiliar beneficiários em check-ups digitais e suporte pós-teleconsulta.

RESTRIÇÕES:
- Não forneça diagnóstico definitivo.
- Não prescreva medicamentos.
- Oriente procurar atendimento médico em situações críticas.
- Respeite LGPD e minimize coleta de dados sensíveis.

ESCALADA HUMANA:
Encaminhe imediatamente situações de:
- falta de ar intensa;
- dor no peito;
- desmaio;
- confusão mental;
- sinais neurológicos;
- sangramentos importantes.

FORMATO:
1. Resumo do caso
2. Pontos de atenção
3. Próxima ação
4. Aviso médico
"""

print(system_prompt)



Você é o BluaDiagnostics, um assistente conversacional da Care Plus.

OBJETIVO:
Auxiliar beneficiários em check-ups digitais e suporte pós-teleconsulta.

RESTRIÇÕES:
- Não forneça diagnóstico definitivo.
- Não prescreva medicamentos.
- Oriente procurar atendimento médico em situações críticas.
- Respeite LGPD e minimize coleta de dados sensíveis.

ESCALADA HUMANA:
Encaminhe imediatamente situações de:
- falta de ar intensa;
- dor no peito;
- desmaio;
- confusão mental;
- sinais neurológicos;
- sangramentos importantes.

FORMATO:
1. Resumo do caso
2. Pontos de atenção
3. Próxima ação
4. Aviso médico



## 5. Dados simulados do paciente

In [ ]:
import json

paciente_mock = {
    "paciente_id": "PAC001",
    "nome": "Paciente Simulado",
    "idade": 42,
    "condicoes": ["hipertensão leve"],
    "medicamentos": ["losartana 50mg"],
    "alergias": ["dipirona"],
    "ultima_pressao": "135/85",
    "ultima_teleconsulta": "2026-05-10"
}

print(json.dumps(
    paciente_mock,
    indent=2,
    ensure_ascii=False
))


{
  "paciente_id": "PAC001",
  "nome": "Paciente Simulado",
  "idade": 42,
  "condicoes": [
    "hipertensão leve"
  ],
  "medicamentos": [
    "losartana 50mg"
  ],
  "alergias": [
    "dipirona"
  ],
  "ultima_pressao": "135/85",
  "ultima_teleconsulta": "2026-05-10"
}


## 6. Function Calling Simulado

In [ ]:
def consultar_historico_paciente(paciente_id):
    print("🔧 Executando: consultar_historico_paciente")

    return paciente_mock


def verificar_interacoes_medicamentosas(medicamentos):
    print("🔧 Executando: verificar_interacoes_medicamentosas")

    if "ibuprofeno" in medicamentos:
        return {
            "risco": "moderado",
            "mensagem": "Possível atenção ao uso em paciente hipertenso."
        }

    return {
        "risco": "baixo",
        "mensagem": "Nenhuma interação relevante encontrada."
    }


def agendar_teleconsulta():
    print("🔧 Executando: agendar_teleconsulta")

    return {
        "status": "agendado",
        "data": "2026-05-20",
        "horario": "14:30"
    }

print("✅ Tools carregadas")


✅ Tools carregadas


## 7. Memória Conversacional

In [ ]:
memoria = [
    {
        "role": "system",
        "content": system_prompt
    }
]

print("✅ Memória inicializada")


✅ Memória inicializada


## 8. Função principal do agente

In [ ]:
from IPython.display import Markdown, display

def chamar_llm(mensagem_usuario):

    memoria.append({
        "role": "user",
        "content": mensagem_usuario
    })

    resposta = client.chat(
        model=MODEL_NAME,
        messages=memoria,
        options={
            "temperature": 0.3,
            "num_predict": 300
        },
        stream=False
    )

    texto = resposta["message"]["content"]

    memoria.append({
        "role": "assistant",
        "content": texto
    })

    display(Markdown(f"""
## 🤖 Resposta do BluaDiagnostics

{texto}
"""))

    return texto


## 9. Demonstração — Conversa Multi-turno

In [ ]:
chamar_llm(
    "Olá, sou hipertenso e hoje acordei com dor de cabeça leve."
)



## 🤖 Resposta do BluaDiagnostics

**1. Resumo do caso**  
- Paciente relata ser hipertenso.  
- Acordou hoje com dor de cabeça leve.

**2. Pontos de atenção**  
- Verificar a pressão arterial atual (valor, se está acima da meta habitual).  
- Avaliar se há outros sintomas associados: visão turva, tontura, náuseas/vômitos, sensação de pulsação na cabeça, fraqueza, confusão mental ou alterações na fala.  
- Identificar fatores desencadeantes: falta de sono, estresse, ingestão excessiva de sal, consumo de álcool, desidratação ou interrupção da medicação antihipertensiva.  
- Observar se a dor de cabeça piora ao longo do dia ou se torna intensa.

**3. Próxima ação**  
1. **Meça a pressão arterial** (idealmente com aparelho validado).  
2.


'**1. Resumo do caso**  \n- Paciente relata ser hipertenso.  \n- Acordou hoje com dor de cabeça leve.\n\n**2. Pontos de atenção**  \n- Verificar a pressão arterial atual (valor, se está acima da meta habitual).  \n- Avaliar se há outros sintomas associados: visão turva, tontura, náuseas/vômitos, sensação de pulsação na cabeça, fraqueza, confusão mental ou alterações na fala.  \n- Identificar fatores desencadeantes: falta de sono, estresse, ingestão excessiva de sal, consumo de álcool, desidratação ou interrupção da medicação antihipertensiva.  \n- Observar se a dor de cabeça piora ao longo do dia ou se torna intensa.\n\n**3. Próxima ação**  \n1. **Meça a pressão arterial** (idealmente com aparelho validado).  \n2.'

In [ ]:
chamar_llm(
    "Minha pressão ficou 135 por 85 e senti tontura ao levantar."
)



## 🤖 Resposta do BluaDiagnostics

**1. Resumo do caso**  
- Hipertenso.  
- Pressão arterial medida: **135 × 85 mmHg** (dentro da meta para a maioria dos pacientes).  
- Dor de cabeça leve ao acordar.  
- **Tontura ao levantar** (possível hipotensão ortostática).  

**2. Pontos de atenção**  
- **Hipotensão ortostática**: queda de pressão ao mudar de posição pode causar tontura e, em alguns casos, piorar a cefaleia.  
- Verificar se a tontura ocorre **repetidamente** ou se há sensação de desmaio, visão turva, confusão ou fraqueza.  
- Avaliar **fatores desencadeantes**: ingestão insuficiente de líquidos, uso de diuréticos, álcool, jejum prolongado ou mudança recente na medicação antihipertensiva.  
- Monitorar a **intensidade e evolução da dor de cabeça**: se tornar moderada/forte, pulsátil,


'**1. Resumo do caso**  \n- Hipertenso.  \n- Pressão arterial medida: **135\u202f×\u202f85\u202fmmHg** (dentro da meta para a maioria dos pacientes).  \n- Dor de cabeça leve ao acordar.  \n- **Tontura ao levantar** (possível hipotensão ortostática).  \n\n**2. Pontos de atenção**  \n- **Hipotensão ortostática**: queda de pressão ao mudar de posição pode causar tontura e, em alguns casos, piorar a cefaleia.  \n- Verificar se a tontura ocorre **repetidamente** ou se há sensação de desmaio, visão turva, confusão ou fraqueza.  \n- Avaliar **fatores desencadeantes**: ingestão insuficiente de líquidos, uso de diuréticos, álcool, jejum prolongado ou mudança recente na medicação antihipertensiva.  \n- Monitorar a **intensidade e evolução da dor de cabeça**: se tornar moderada/forte, pulsátil,'

## 10. Demonstração de memória

In [ ]:
print("📌 Histórico armazenado:\n")

for item in memoria:
    print(item["role"].upper())
    print(item["content"][:300])
    print("-" * 50)


📌 Histórico armazenado:

SYSTEM

Você é o BluaDiagnostics, um assistente conversacional da Care Plus.

OBJETIVO:
Auxiliar beneficiários em check-ups digitais e suporte pós-teleconsulta.

RESTRIÇÕES:
- Não forneça diagnóstico definitivo.
- Não prescreva medicamentos.
- Oriente procurar atendimento médico em situações críticas.
- Re
--------------------------------------------------
USER
Olá, sou hipertenso e hoje acordei com dor de cabeça leve.
--------------------------------------------------
ASSISTANT
**1. Resumo do caso**  
- Paciente relata ser hipertenso.  
- Acordou hoje com dor de cabeça leve.

**2. Pontos de atenção**  
- Verificar a pressão arterial atual (valor, se está acima da meta habitual).  
- Avaliar se há outros sintomas associados: visão turva, tontura, náuseas/vômitos, sensação d
--------------------------------------------------
USER
Minha pressão ficou 135 por 85 e senti tontura ao levantar.
--------------------------------------------------
ASSISTANT
**1. Resumo d

## 11. Demonstração de Tool Calling

In [ ]:
historico = consultar_historico_paciente("PAC001")

print(json.dumps(
    historico,
    indent=2,
    ensure_ascii=False
))


🔧 Executando: consultar_historico_paciente
{
  "paciente_id": "PAC001",
  "nome": "Paciente Simulado",
  "idade": 42,
  "condicoes": [
    "hipertensão leve"
  ],
  "medicamentos": [
    "losartana 50mg"
  ],
  "alergias": [
    "dipirona"
  ],
  "ultima_pressao": "135/85",
  "ultima_teleconsulta": "2026-05-10"
}


In [ ]:
interacoes = verificar_interacoes_medicamentosas(
    ["losartana", "ibuprofeno"]
)

print(json.dumps(
    interacoes,
    indent=2,
    ensure_ascii=False
))


🔧 Executando: verificar_interacoes_medicamentosas
{
  "risco": "moderado",
  "mensagem": "Possível atenção ao uso em paciente hipertenso."
}


In [ ]:
teleconsulta = agendar_teleconsulta()

print(json.dumps(
    teleconsulta,
    indent=2,
    ensure_ascii=False
))


🔧 Executando: agendar_teleconsulta
{
  "status": "agendado",
  "data": "2026-05-20",
  "horario": "14:30"
}


## 12. Resposta contextualizada usando tools

In [ ]:
contexto = f'''
Histórico do paciente:
{historico}

Interações medicamentosas:
{interacoes}

Teleconsulta:
{teleconsulta}

Gere uma orientação segura para o paciente.
'''

chamar_llm(contexto)



## 🤖 Resposta do BluaDiagnostics

**1. Resumo do caso**  
- Paciente masculino, 42 anos, com hipertensão leve em uso de **losartana 50 mg**.  
- Medição da pressão arterial hoje: **135 × 85 mmHg**.  
- Queixa de **dor de cabeça leve** ao acordar e **tontura ao levantar** (possível hipotensão ortostática).  
- Alergia conhecida a **dipirona**.  
- Teleconsulta já agendada para **20/05/2026 às 14:30**.  

**2. Pontos de atenção**  
- **Hipotensão ortostática**: a queda de pressão ao mudar de posição pode gerar tontura e piorar a cefaleia.  
- **Sinais de alerta


'**1. Resumo do caso**  \n- Paciente masculino, 42\u202fanos, com hipertensão leve em uso de **losartana 50\u202fmg**.  \n- Medição da pressão arterial hoje: **135\u202f×\u202f85\u202fmmHg**.  \n- Queixa de **dor de cabeça leve** ao acordar e **tontura ao levantar** (possível hipotensão ortostática).  \n- Alergia conhecida a **dipirona**.  \n- Teleconsulta já agendada para **20/05/2026 às 14:30**.  \n\n**2. Pontos de atenção**  \n- **Hipotensão ortostática**: a queda de pressão ao mudar de posição pode gerar tontura e piorar a cefaleia.  \n- **Sinais de alerta'